In [ ]:
%matplotlib widget

import sympy as sp
import numpy as np
import matplotlib.pyplot as plt

from ipywidgets import (
    IntSlider, HTML, HTMLMath,
    VBox, HBox, Layout
)
from IPython.display import display

plt.ioff()

# ============================================================
# STYLE
# ============================================================

display(HTML("""
<style>
.container{width:98%!important;max-width:none!important}
.output_area,.output_subarea,.jp-Cell-outputWrapper,.jp-OutputArea,
.jp-OutputArea-child,.jp-OutputArea-output,.widget-output,
.jupyter-widgets-output-area,.widget-box{
max-width:none!important;height:auto!important;max-height:none!important;overflow:visible!important}
.output_scroll{height:auto!important;max-height:none!important;overflow:visible!important;box-shadow:none!important}
.jupyter-matplotlib,.jupyter-matplotlib-figure{overflow:visible!important;resize:none!important}

.par-title{font-family:Arial;font-size:20px;font-weight:bold;color:#6f3fa0}
.par-label{font-family:Arial;font-size:14px;font-weight:bold}
.par-value{font-family:Arial;font-size:14px;font-weight:bold;color:#0b3d91}
</style>
"""))

# ============================================================
# DOCUMENTATION
# ============================================================

documentation = HTML("""
<div style="width:1140px;padding:10px 14px;border:1px solid #d2c2df;
font-family:Arial;font-size:15px;line-height:1.5;box-sizing:border-box">

<div class="par-title" style="margin-bottom:8px">
Bessel Inequality and Parseval Identity
</div>

<div style="margin-bottom:6px">
For an orthonormal set {φₙ}, Bessel's inequality states that the energy
contained in any finite set of projection coefficients cannot exceed the
total signal energy.
</div>

<div style="margin-bottom:6px;text-align:center">
<b>Σ |〈x,φₙ〉|² ≤ ‖x‖².</b>
</div>

<div style="margin-bottom:6px">
If the orthonormal set is complete and all coefficients are included,
the inequality becomes Parseval's identity.
</div>

<div>
<b>This notebook:</b> calculates the coefficients symbolically and shows
how the retained coefficient energy approaches the total signal energy
as the number of Fourier modes increases.
</div>
</div>
""")

# ============================================================
# SYMBOLS
# ============================================================

t = sp.symbols('t', real=True)

# Rectangular pulse:
# x(t)=1 for |t|<=pi/2, 0 elsewhere

x_piecewise = sp.Piecewise(
    (1,sp.Abs(t)<=sp.pi/2),
    (0,True)
)

# Total energy, calculated symbolically
total_energy = sp.integrate(
    1,
    (t,-sp.pi/2,sp.pi/2)
)

# ============================================================
# ORTHONORMAL TRIGONOMETRIC BASIS
#
# phi_0 = 1/sqrt(2*pi)
#
# cos(nt)/sqrt(pi)
# sin(nt)/sqrt(pi)
# ============================================================

phi0 = 1/sp.sqrt(2*sp.pi)

c0 = sp.simplify(
    sp.integrate(
        phi0,
        (t,-sp.pi/2,sp.pi/2)
    )
)

# ============================================================
# CONTROLS
# ============================================================

N_slider = IntSlider(
    min=1,max=25,step=1,value=5,
    readout=False,
    continuous_update=True,
    layout=Layout(width='300px')
)

N_value = HTML()

control_row = HBox([
    HTML(
        '<div class="par-label">Number of harmonics N:</div>',
        layout=Layout(width='190px')
    ),
    N_slider,
    N_value
],layout=Layout(width='560px',height='40px',align_items='center'))

controls = VBox([
    HTML('<div class="par-title" style="margin-bottom:6px">Parameters</div>'),
    control_row
],layout=Layout(
    width='600px',
    padding='10px 14px',
    border='1px solid #d2c2df'
))

# ============================================================
# RESULT PANEL
# ============================================================

result_math = HTMLMath()
result_text = HTML()

result_panel = VBox([
    HTML('<div class="par-title" style="margin-bottom:7px">Current Energy Balance</div>'),
    result_math,
    result_text
],layout=Layout(
    width='520px',
    padding='10px 14px',
    border='1px solid #d2c2df'
))

top_row = HBox(
    [controls,result_panel],
    layout=Layout(width='1135px',gap='15px',align_items='stretch')
)

# ============================================================
# PRECOMPUTE SYMBOLIC COEFFICIENTS
# ============================================================

MAX_N = 25

cos_coeffs = []
sin_coeffs = []

for n in range(1,MAX_N+1):

    phi_c = sp.cos(n*t)/sp.sqrt(sp.pi)
    phi_s = sp.sin(n*t)/sp.sqrt(sp.pi)

    cn = sp.simplify(
        sp.integrate(
            phi_c,
            (t,-sp.pi/2,sp.pi/2)
        )
    )

    sn = sp.simplify(
        sp.integrate(
            phi_s,
            (t,-sp.pi/2,sp.pi/2)
        )
    )

    cos_coeffs.append(cn)
    sin_coeffs.append(sn)

# ============================================================
# NUMERICAL SIGNAL
# ============================================================

tt = np.linspace(-np.pi,np.pi,1800)

x_num = (
    np.abs(tt)<=np.pi/2
).astype(float)

# ============================================================
# FIGURE 1 — RECONSTRUCTION
# ============================================================

fig1,ax1 = plt.subplots(figsize=(7.1,4.8))

fig1.canvas.header_visible=False
fig1.canvas.footer_visible=False
fig1.canvas.toolbar_visible=False
fig1.canvas.layout=Layout(width='710px',height='480px',margin='0px')

ax1.set_title(
    'Fourier Reconstruction',
    fontsize=14,fontweight='bold',color='#6f3fa0'
)

ax1.set_xlabel('t')
ax1.set_ylabel('Amplitude')

ax1.set_xlim(-np.pi,np.pi)
ax1.set_ylim(-0.35,1.35)

ax1.set_xticks([-np.pi,-np.pi/2,0,np.pi/2,np.pi])
ax1.set_xticklabels([
    r'$-\pi$',r'$-\pi/2$','0',r'$\pi/2$',r'$\pi$'
])

ax1.grid(True,linestyle=':',alpha=.4)

original_line, = ax1.plot(
    tt,
    x_num,
    linestyle='--',
    linewidth=1.5,
    label='Original x(t)'
)

reconstruction_line, = ax1.plot(
    tt,
    np.zeros_like(tt),
    linewidth=2.2,
    label='Partial expansion'
)

ax1.legend(loc='upper right')

fig1.subplots_adjust(
    left=.10,right=.97,
    top=.89,bottom=.14
)

# ============================================================
# FIGURE 2 — RETAINED ENERGY
# ============================================================

fig2,ax2 = plt.subplots(figsize=(4.3,4.8))

fig2.canvas.header_visible=False
fig2.canvas.footer_visible=False
fig2.canvas.toolbar_visible=False
fig2.canvas.layout=Layout(width='430px',height='480px',margin='0px')

ax2.set_title(
    'Bessel → Parseval',
    fontsize=14,fontweight='bold',color='#0b3d91'
)

ax2.set_xlabel('Number of harmonics N')
ax2.set_ylabel('Retained energy')

ax2.set_xlim(1,MAX_N)
ax2.set_ylim(0,float(total_energy)*1.08)

ax2.grid(True,linestyle=':',alpha=.4)

N_axis = np.arange(1,MAX_N+1)

energy_curve, = ax2.plot(
    N_axis,
    np.zeros(MAX_N),
    linewidth=2,
    marker='o',
    markersize=3,
    label='Retained energy'
)

total_line = ax2.axhline(
    float(total_energy),
    linestyle='--',
    linewidth=1.4,
    label='Total energy'
)

current_marker, = ax2.plot(
    [],
    [],
    linestyle='None',
    marker='o',
    markersize=8
)

ax2.legend(loc='lower right',fontsize=8)

fig2.subplots_adjust(
    left=.16,right=.97,
    top=.89,bottom=.14
)

figures_row = HBox(
    [fig1.canvas,fig2.canvas],
    layout=Layout(width='1145px',gap='10px',align_items='flex-start')
)

# ============================================================
# PRECOMPUTE ENERGY CURVE
# ============================================================

partial_energies = []

for N in range(1,MAX_N+1):

    energy = sp.simplify(
        c0**2
        +
        sum(
            cos_coeffs[n-1]**2
            +
            sin_coeffs[n-1]**2
            for n in range(1,N+1)
        )
    )

    partial_energies.append(
        float(sp.N(energy))
    )

energy_curve.set_ydata(
    partial_energies
)

# ============================================================
# UPDATE
# ============================================================

def update(change=None):

    N=N_slider.value

    N_value.value=(
        f'<div class="par-value">{N}</div>'
    )

    # --------------------------------------------------------
    # Symbolic retained energy
    # --------------------------------------------------------

    retained_symbolic = sp.simplify(
        c0**2
        +
        sum(
            cos_coeffs[n-1]**2
            +
            sin_coeffs[n-1]**2
            for n in range(1,N+1)
        )
    )

    retained_numeric = float(
        sp.N(retained_symbolic)
    )

    fraction = (
        retained_numeric
        /
        float(total_energy)
    )

    # --------------------------------------------------------
    # Reconstruction
    # --------------------------------------------------------

    reconstruction = (
        float(c0/sp.sqrt(2*sp.pi))
        *
        np.ones_like(tt)
    )

    for n in range(1,N+1):

        cn = float(
            sp.N(cos_coeffs[n-1])
        )

        sn = float(
            sp.N(sin_coeffs[n-1])
        )

        reconstruction += (
            cn
            *
            np.cos(n*tt)
            /
            np.sqrt(np.pi)
        )

        reconstruction += (
            sn
            *
            np.sin(n*tt)
            /
            np.sqrt(np.pi)
        )

    reconstruction_line.set_ydata(
        reconstruction
    )

    current_marker.set_data(
        [N],
        [retained_numeric]
    )

    # --------------------------------------------------------
    # Symbolic output
    # --------------------------------------------------------

    result_math.value = (
        r'\('
        r'\|x\|^2='
        +
        sp.latex(total_energy)
        +
        r'\)'
        r'<div style="height:8px"></div>'
        r'\('
        r'E_N='
        r'\sum_{k=0}^{N}|c_k|^2='
        +
        sp.latex(retained_symbolic).replace(r'\frac',r'\dfrac')
        +
        r'\)'
        r'<div style="height:8px"></div>'
        r'\('
        r'E_N\leq\|x\|^2'
        r'\)'
    )

    result_text.value = (
        '<div style="font-family:Arial;font-size:14px;line-height:1.5;margin-top:8px">'
        f'<b>Energy retained:</b> {100*fraction:.2f}%'
        '</div>'
    )

    fig1.canvas.draw_idle()
    fig2.canvas.draw_idle()

N_slider.observe(
    update,
    names='value'
)

update()

# ============================================================
# INTERPRETATION
# ============================================================

interpretation = HTML("""
<div style="width:1140px;padding:11px 14px;border:1px solid #d7c7e5;
font-family:Arial;font-size:14px;line-height:1.58;box-sizing:border-box;margin-top:6px">

<div style="color:#6f3fa0;font-size:17px;font-weight:bold;margin-bottom:7px">
Interpretation
</div>

<div style="margin-bottom:6px">
For a finite number of orthonormal basis functions, the sum of squared
projection coefficients is smaller than or equal to the total signal energy.
This is Bessel's inequality.
</div>

<div style="margin-bottom:6px">
As more basis functions are retained, the partial Fourier expansion improves
and the coefficient-energy sum increases monotonically toward the total energy.
</div>

<div>
For a complete orthonormal basis and in the limit N→∞, no energy is missing:
the Bessel inequality becomes Parseval's identity.
</div>

</div>
""")

display(
    VBox(
        [
            documentation,
            top_row,
            figures_row,
            interpretation
        ],
        layout=Layout(
            width='1180px',
            gap='8px',
            align_items='flex-start'
        )
    )
)